In [11]:
# Import libraries
import tensorflow as tf
import tensorflow_recommenders as tfrs
import pandas as pd
import numpy as np
from typing import Dict, Text

# Load datasets
ratings_df = pd.read_csv("ratings.csv")
books_df = pd.read_csv("books.csv")

# Ensure consistent types. Strings is necessary because TensorFlow expects strings, and Ids like bookID or userID may come in as integers
ratings_df["book_id"] = ratings_df["book_id"].astype(str)
ratings_df["user_id"] = ratings_df["user_id"].astype(str)
books_df["book_id"] = books_df["book_id"].astype(str)

# Filter ratings to valid books. This step is necessary as a data cleaning step. Some of the book_ids in the ratings file are not in the books file
valid_book_ids = set(books_df["book_id"])
ratings_df = ratings_df[ratings_df["book_id"].isin(valid_book_ids)]

# Normalize Ratings by User Mean. This step is important because it allows the model to learn which items users prefer relative to the user's ratings. 

# Create the user_mean_ratings DataFrame by grouping the ratings_df by user, calculating the mean for each user, and creating a "user_mean" column for each user. Reset_index() is then used to ensure the output is a DataFrame
user_mean_ratings = ratings_df.groupby("user_id")["rating"].mean().rename("user_mean").reset_index()
# Merge the newly created user_mean_ratings dataframe with the ratings_df and save as ratings_df
ratings_df = ratings_df.merge(user_mean_ratings, on="user_id")
# Create a normalized_rating column that subtracts the rating for a specific book from the user's mean rating
ratings_df["normalized_rating"] = ratings_df["rating"] - ratings_df["user_mean"]

# Convert data to a TensorFlow dataset. TensorFlow cannot use Pandas DataFrames. from_tensor_slices converts each row of the dataframe to an element in the TensorFlow dataset
ratings = tf.data.Dataset.from_tensor_slices({
    "user_id": ratings_df["user_id"].values,
    "book_id": ratings_df["book_id"].values,
    "rating": ratings_df["normalized_rating"].values.astype("float32"),
})

# Build vocabularies
user_ids = ratings_df["user_id"].unique()
book_ids = ratings_df["book_id"].unique()

user_vocab = tf.keras.layers.StringLookup(vocabulary=user_ids, mask_token=None)
book_vocab = tf.keras.layers.StringLookup(vocabulary=book_ids, mask_token=None)

# Create Book Ranking Model Class. This class creates embeddings and develops models
class BookRankingModel(tf.keras.Model):
    def __init__(self, user_vocab, book_vocab):
        super().__init__()
        embedding_dim = 32
        # Created embeddings for users 
        self.user_embedding = tf.keras.Sequential([
            user_vocab,
            tf.keras.layers.Embedding(
                len(user_vocab.get_vocabulary()),
                embedding_dim,
                embeddings_regularizer=tf.keras.regularizers.l2(1e-6)
            )
        ])
        # Create embeddings for books
        self.book_embedding = tf.keras.Sequential([
            book_vocab,
            tf.keras.layers.Embedding(
                len(book_vocab.get_vocabulary()),
                embedding_dim,
                embeddings_regularizer=tf.keras.regularizers.l2(1e-6)
            )
        ])
        # Create a model that uses embeddings to predict ratings for a specified user. A dropout layer is included to address overfitting concerns
        self.rating_predictor = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation="relu"),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(1)  
        ])

    # 
    def call(self, features):
        user_emb = self.user_embedding(features["user_id"])
        book_emb = self.book_embedding(features["book_id"])
        x = tf.concat([user_emb, book_emb], axis=1)
        return self.rating_predictor(x)

# Create class to specify model training logic
class BookRankingModelTFRS(tfrs.models.Model):
    def __init__(self, user_vocab, book_vocab):
        super().__init__()
        # Initiate model defined in class above
        self.ranking_model = BookRankingModel(user_vocab, book_vocab)
        # Specifies task to be performed is a ranking task with loss MSE and metric RMSE
        self.task = tfrs.tasks.Ranking(
            loss=tf.keras.losses.MeanSquaredError(),
            metrics=[tf.keras.metrics.RootMeanSquaredError()]
        )
    # Creates the compute_loss method used to measure metrics and optimize model 
    def compute_loss(self, features: Dict[Text, tf.Tensor], training=False) -> tf.Tensor:
        labels = features.pop("rating")
        predictions = self.ranking_model(features)
        return self.task(labels=labels, predictions=predictions)

# Splits dataset into batches of 256. .cache() caches the dataset in memory after the first epoch, which allows the model to run more quickly
full_dataset = ratings.batch(256).cache()

# Create model from BookRankingModelTFRS class
model = BookRankingModelTFRS(user_vocab, book_vocab)
# Compile the model
model.compile()
# Train the model using the full_dataset and 5 epochs
model.fit(full_dataset, epochs=5)

# Prediction and Recommendation

# Book ID to title lookup. Match each book_id to it's corresponding title so that recommendations are titles and not book_ids
book_lookup = dict(zip(books_df["book_id"], books_df["title"]))

# ALlow user to input user_id to see recommendations for that user
sample_user = input("Enter a user_id: ")
print(f"\nTop Book Recommendations for User {sample_user}:")

# Get mean rating for this user, which is used to personalize recommendations in relation to the user's mean rating
user_mean = user_mean_ratings[user_mean_ratings["user_id"] == sample_user]["user_mean"].values
if len(user_mean) == 0:
    print("User not found.")
else:
    user_mean = user_mean[0]

    # Create a datset of books and batches 
    book_ids_ds = tf.data.Dataset.from_tensor_slices(book_ids).batch(128)
    predictions = []
    # Loop through each batch
    for batch in book_ids_ds:
        # Creates a tensor of repeating user ids that matches the user's id to each book in the batch
        batch_user = tf.constant([sample_user] * batch.shape[0])
        # Create dictionary of user id, whcih is the user id, and book_id, which is the book id. This is the format the model is expecting
        features = {"user_id": batch_user, "book_id": batch}
        # Use the ranking_model to predict scores for the user. This will predict normalized scores
        scores = model.ranking_model(features).numpy().flatten()

        # Add user's mean back to normalized prediction and ensure a minimum score of 0 and maximum of 5
        restored_scores = np.clip(scores + user_mean, 0, 5)
        # Add prediction to empty predictions list
        predictions.extend(zip(batch.numpy(), restored_scores))

    # Sort by predicted rating in descending order
    top_predictions = sorted(predictions, key=lambda x: -x[1])[:5]

    # Show recommendations
    for book_id_bytes, predicted_rating in top_predictions:
        book_id_str = book_id_bytes.decode("utf-8")
        title = book_lookup.get(book_id_str, "Unknown Title")
        print(f"- {title} (Predicted Rating: {predicted_rating:.2f})")



Epoch 1/5
312/312 [==============================] - 10s 16ms/step - root_mean_squared_error: 0.6697 - loss: 0.4461 - regularization_loss: 7.9346e-04 - total_loss: 0.4469
Epoch 2/5
312/312 [==============================] - 5s 15ms/step - root_mean_squared_error: 0.6592 - loss: 0.4323 - regularization_loss: 7.9671e-04 - total_loss: 0.4331
Epoch 3/5
312/312 [==============================] - 5s 15ms/step - root_mean_squared_error: 0.6552 - loss: 0.4270 - regularization_loss: 8.0259e-04 - total_loss: 0.4278
Epoch 4/5
312/312 [==============================] - 5s 16ms/step - root_mean_squared_error: 0.6489 - loss: 0.4189 - regularization_loss: 8.1155e-04 - total_loss: 0.4197
Epoch 5/5
312/312 [==============================] - 5s 16ms/step - root_mean_squared_error: 0.6356 - loss: 0.4019 - regularization_loss: 8.2413e-04 - total_loss: 0.4027


Enter a user_id:  1274



Top Book Recommendations for User 1274:
- Fall on Your Knees (Predicted Rating: 4.16)
- White Teeth (Predicted Rating: 4.15)
- The Plot Against America (Predicted Rating: 4.13)
- Maurice (Predicted Rating: 4.13)
- West with the Night (Predicted Rating: 4.12)


In [ ]:
#The code below is used to genreated the csv file that is uploaded to the website created with vibe coding
# Import libraries 
import random
import pandas as pd
import numpy as np

# Sample 50 unique users
sample_users = np.random.choice(ratings_df["user_id"].unique(), size=50, replace=False)

# Print sampled user IDs
print("👤 Sampled User IDs:\n", sample_users)
# Create a datset of books and batches
book_ids_ds = tf.data.Dataset.from_tensor_slices(book_ids).batch(128)
# Create empty recommendations list
recommendations = []
# loop through 50 sampled users
for user_id in sample_users:
    # Get mean rating for each user
    user_mean = user_mean_ratings[user_mean_ratings["user_id"] == user_id]["user_mean"].values
    if len(user_mean) == 0:
        continue
    user_mean = user_mean[0]
    # Create empty predictions list for the specific user
    user_predictions = []
    # Loop through each batch
    for batch in book_ids_ds:
        # Creates a tensor of repeating user ids that matches the user's id to each book in the batch
        batch_user = tf.constant([user_id] * batch.shape[0])
        # Create dictionary of user id, whcih is the user id, and book_id, which is the book id. This is the format the model is expecting
        features = {"user_id": batch_user, "book_id": batch}
        # Use the ranking_model to predict scores for the user. This will predict normalized scores
        scores = model.ranking_model(features).numpy().flatten()

        # Add user's mean back to normalized prediction and ensure a minimum score of 0 and maximum of 5
        restored_scores = np.clip(scores + user_mean, 0, 5)
        # Update the list of predictions back to the list for this specific user
        user_predictions.extend(zip(batch.numpy(), restored_scores))

    # Get top 5 recommendations 
    top_books = sorted(user_predictions, key=lambda x: -x[1])[:5]
    # Show data
    for book_id_bytes, score in top_books:
        book_id_str = book_id_bytes.decode("utf-8")

        # Lookup metadata about the recommended book
        book_data = books_df[books_df["book_id"] == book_id_str]
        title = book_data["title"].values[0] if not book_data.empty else "Unknown Title"
        image_url = (
            book_data["image_url"].values[0]
            if "image_url" in book_data.columns and not book_data.empty
            else "https://via.placeholder.com/150"
        )

        recommendations.append({
            "user_id": user_id,
            "book_id": book_id_str,
            "title": title,
            "predicted_rating": round(float(score), 4),
            "image_url": image_url
        })

# Save to CSV
df_recs = pd.DataFrame(recommendations)
df_recs.to_csv("Book_Recommendations_sample50.csv", index=False)

print("✅ CSV with image URLs saved as 'Book_Recommendations_sample50.csv'")
